<a href="https://colab.research.google.com/github/yiii1014/114-2-/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q google-generativeai

In [14]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [15]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-flash')

In [16]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Moh2bKPmrs-PWC_C_7h9F5JAatitPm7LV171nM7ilz8/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [17]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [18]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    model = genai.GenerativeModel('gemini-2.5-flash')

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [20]:
new_grades

[['2026-03-28', '英文', 85]]

In [21]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的成績資料所做的摘要與常見迷思整理：\n\n---\n\n### 成績摘要\n\n*   **日期：** 2026年3月28日\n*   **科目：** 英文\n*   **成績：** 85分\n\n這份資料顯示了一位學生在2026年3月28日英文科目測驗中，獲得了85分。這是一個單一科目、單一時間點的成績記錄。\n\n### 常見迷思整理（關於單一成績的解讀）\n\n1.  **迷思一：單一成績代表學生的全部能力或學習態度。**\n    *   **澄清：** 一個85分是學生在特定時間、特定測驗範圍內的表現。它無法全面反映學生在該科目所有的能力面向（例如口說、聽力、寫作的流暢度等），也無法判斷其學習過程中的努力程度、進步幅度或對該科目的整體興趣。學生可能在這次測驗中表現出色，也可能只是特定單元的強項。\n\n2.  **迷思二：單一高分（或低分）能預測學生未來的表現。**\n    *   **澄清：** 學習是一個動態的過程。學生的表現會受到多種因素影響，包括教材難度、學習方法、個人狀態、準備時間、測驗題型等。一個85分的成績，無論其高低，都無法絕對預示學生在未來類似測驗中的表現，它只是一個當下的快照。\n\n3.  **迷思三：僅憑成績數字就能判斷「好」或「不好」。**\n    *   **澄清：** 成績數字本身是缺乏情境的。85分在沒有其他參考數據（如班級平均成績、試卷難度、老師的評分標準、學生的個人學習目標或過往成績）的情況下，很難進行客觀的「好」或「不好」的判斷。對於某些學生而言，85分可能是巨大的進步；對於另一些學生，則可能未達其預期。\n\n4.  **迷思四：成績完全等同於學習成效或知識掌握度。**\n    *   **澄清：** 成績是衡量學習成效的一種方式，但並非唯一或絕對的方式。有時，成績可能受限於測驗形式，未能充分評估學生對知識的深入理解、應用能力或解決問題的策略。學生可能考了85分，但對於某些概念仍有模糊之處，也可能因為粗心而失分。\n\n---'

In [22]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 將完整的 AI 摘要寫入單一儲存格
        ws.update_cell(next_row, 3, summary)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：化學
請輸入 化學 的成績：75
已記錄：日期: 2026-03-28, 科目: 化學, 成績: 75

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，針對您提供的學生單一成績，以下是一個簡單的摘要與常見迷思整理：

---

### 成績摘要

這份資料呈現了一位學生在**2026年3月28日**進行的**化學科目**評估中，獲得的**75分**。這是一個單一的數據點，代表了該學生在特定時間點，針對某項化學知識或技能評估的結果。

---

### 常見迷思整理（基於單一成績的解讀）

在只有一個成績數據點的情況下，人們常會產生一些誤解，以下列出幾點：

1.  **迷思一：單一成績能全面評估學生能力或學習全貌。**
    *   **總結：** 一次性的75分僅代表學生在特定時間、特定評估方式下的表現。它無法反映學生在其他化學領域的知識、學習過程中的努力、思考能力、實踐技能，或是整體學業表現。學生在不同情境下可能會展現出不同的能力水平。

2.  **迷思二：75分代表「好」或「不好」的絕對判斷。**
    *   **總結：** 分數的意義高度依賴情境。例如，如果這是一次極具挑戰性的考試，班級平均分可能很低，則75分可能代表相對優秀的表現；反之，若考試難度偏低，或評分標準較為寬鬆，75分可能還有進步空間。離開上下文（如課程目標、試卷難度、班級平均、學生個人進步軌跡），分數本身只是個數字。

3.  **迷思三：單一成績能準確預測未來表現。**
    *   **總結：** 這次75分的表現是學習過程中的一個快照，而非永久性的預測。學生的表現會隨著學習策略、投入時間、健康狀況、教學方式甚至心態而變化。一次成績並非固定的未來表現指標。

4.  **迷思四：分數直接揭示了學生學習上的「為何」或「如何」。**
  

In [23]:
if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：q
沒有輸入任何成績，程式結束。
